In [1]:
import os
os.environ["HF_HOME"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf"
os.environ["HF_HUB_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers"
os.environ["HF_HUB_OFFLINE"] = "1"   # hard offline, raises if not found locally

from transformers import AutoProcessor, SiglipVisionModel

model_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(model_id, local_files_only=True)
vision    = SiglipVisionModel.from_pretrained(model_id, local_files_only=True)


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
import torch, pandas as pd, json
from PIL import Image, UnidentifiedImageError
from pathlib import Path
from transformers import AutoProcessor, AutoModel

MODEL = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()

DATA_CSV = Path("kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated.csv")
SCHEMA_JSON = DATA_CSV.parent / "labels_schema_filtered.json"
labels = json.loads(SCHEMA_JSON.read_text())
df = pd.read_csv(DATA_CSV)

PROMPTS = {
    "qr": "a black and white QR code square",
    "logo": "a simple company or journal logo icon on a page",
    "journal_banner": "the header banner of a scientific journal article page",
    "pure_text": "a page of only text paragraphs without pictures",
    "book_cover": "a book or cover page with a large title",
    "empty": "an almost blank white rectangle with no content",
    "ecg": "an ECG electrocardiogram waveform plot",
    "profile": "a portrait headshot profile photo of a person",
    "gel_electrophoresis": "a gel electrophoresis image with lanes and bands",
    "map": "a geographic map with regions, borders or roads",
    "tiny": "a very small icon or tiny image with little detail",
    "house": "a house or building exterior photo or drawing",
    "illustration": "a simple illustration or drawing graphic",
    "phylogenetic_tree": "a phylogenetic tree diagram of species relationships",
    "doc_fullpage": "a full scanned document page with margins and text blocks",
    "chart": "a data chart or plot such as bar or line chart",
}
def label_to_prompt(lbl:str)->str:
    return PROMPTS.get(lbl, f"an image containing {lbl.replace('_',' ')}")

# text embeddings
text_inputs = processor(text=[label_to_prompt(l) for l in labels],
                        return_tensors="pt", padding=True, truncation=True)
with torch.no_grad():
    text_emb = model.get_text_features(**text_inputs)
    text_emb = torch.nn.functional.normalize(text_emb, dim=-1)

MIN_SIDE = 16  # skip or upscale tiny bitmaps

def score_image(path: str):
    try:
        im = Image.open(path).convert("RGB")
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None
    w, h = im.size
    if min(w, h) < MIN_SIDE:
        # Option A: skip
        # return None
        # Option B: lightly upscale so transforms behave
        scale = MIN_SIDE / max(1, min(w, h))
        im = im.resize((max(MIN_SIDE, int(w*scale)), max(MIN_SIDE, int(h*scale))), Image.BILINEAR)

    # IMPORTANT: pass PIL.Image directly; don't pass numpy arrays
    inputs = processor(images=im, return_tensors="pt")
    with torch.no_grad():
        img_emb = model.get_image_features(**inputs)
        img_emb = torch.nn.functional.normalize(img_emb, dim=-1)
        sim = img_emb @ text_emb.T
    return sim.squeeze(0)

# sample and evaluate
sample = df[df[[f"label_{l}" for l in labels]].sum(axis=1) >= 1]\
           .sample(min(24, len(df)), random_state=0)

hit1 = {l:0 for l in labels}
hit5 = {l:0 for l in labels}
cnt  = {l:0 for l in labels}

for _, r in sample.iterrows():
    sims = score_image(r["path"])
    if sims is None:
        print("⚠️ skipped:", r["path"])
        continue
    top = sims.argsort(descending=True).tolist()
    top5_labels = [labels[i] for i in top[:5]]
    gt = [l for l in labels if int(r[f"label_{l}"])==1]
    print("—", r["path"])
    print("  GT:", gt)
    print("  Top-5:", [f"{labels[i]}:{float(sims[i]):.3f}" for i in top[:5]])
    for g in gt:
        cnt[g]+=1
        if labels[top[0]] == g: hit1[g]+=1
        if g in top5_labels: hit5[g]+=1

print("\nPer-class Hit@1 / Hit@5 on sample (zero-shot):")
for l in labels:
    if cnt[l]>0:
        print(f"{l:22s}: H@1={hit1[l]}/{cnt[l]}  H@5={hit5[l]}/{cnt[l]}")


— /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images/1-case-A case report of cutaneous leishmaniasis/p0001_xref14.png
  GT: ['logo']
  Top-5: ['logo:0.051', 'journal_banner:0.040', 'profile:0.034', 'book_cover:0.027', 'pure_text:0.026']
— /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images/1-case-Complicated cutaneous leishmaniasis caused by an imported case of Leishmania tropica in Japan_ a case report/p0005_xref263.png
  GT: ['pure_text']
  Top-5: ['book_cover:0.121', 'journal_banner:0.116', 'pure_text:0.102', 'doc_fullpage:0.095', 'logo:0.084']
— /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images/1-case-Atypical Cutaneous Leishmaniasis by Leishmania mexicana_ A Case Report with Dermoscopic, Histopathological and Molecular Study/p0004_xref20.jpeg
  GT: ['gel_electrophoresis']
  Top-5: ['gel_electrophoresis:0.116', 'doc_fullpage:0.059', 'journal_banner:0.050', 'ecg:0.048', 'empty:0.046']
— /media/pc1/Ubuntu/Exte

In [4]:
# assumes you already built: processor, model, labels, df, text_emb, score_image(...)
import torch
import numpy as np
from collections import defaultdict

rows = df[df[[f"label_{l}" for l in labels]].sum(axis=1) >= 1]
hit1 = defaultdict(int)
hit5 = defaultdict(int)
cnt = defaultdict(int)

for _, r in rows.iterrows():
    sims = score_image(r["path"])
    if sims is None: 
        continue
    sims = sims.cpu() if hasattr(sims, "cpu") else sims
    
    # Fixed: Use PyTorch argsort to avoid NumPy deprecation warning
    if isinstance(sims, torch.Tensor):
        order = sims.argsort(descending=True).tolist()
    else:
        # Fallback for non-tensor sims
        v = np.asarray(sims)
        order = np.argsort(-v).tolist()
    
    top5 = [labels[i] for i in order[:5]]
    gt = [l for l in labels if int(r[f"label_{l}"]) == 1]
    
    for g in gt:
        cnt[g] += 1
        if labels[order[0]] == g: 
            hit1[g] += 1
        if g in top5: 
            hit5[g] += 1

print("\nZero-shot Hit@1 / Hit@5 (full set):")
micro_h1_num = micro_h5_num = micro_den = 0

for l in labels:
    if cnt[l]:
        print(f"{l:22s}: H@1={hit1[l]}/{cnt[l]}  H@5={hit5[l]}/{cnt[l]}")
        micro_h1_num += hit1[l]
        micro_h5_num += hit5[l] 
        micro_den += cnt[l]

print(f"\nMicro H@1 = {micro_h1_num}/{micro_den} = {micro_h1_num/max(1,micro_den):.3f}")
print(f"Micro H@5 = {micro_h5_num}/{micro_den} = {micro_h5_num/max(1,micro_den):.3f}")


Zero-shot Hit@1 / Hit@5 (full set):
qr                    : H@1=8/10  H@5=8/10
logo                  : H@1=26/35  H@5=35/35
chart                 : H@1=1/2  H@5=2/2
journal_banner        : H@1=9/36  H@5=32/36
pure_text             : H@1=0/8  H@5=7/8
book_cover            : H@1=0/2  H@5=2/2
empty                 : H@1=16/38  H@5=22/38
ecg                   : H@1=1/1  H@5=1/1
profile               : H@1=4/4  H@5=4/4
gel_electrophoresis   : H@1=7/7  H@5=7/7
map                   : H@1=1/5  H@5=1/5
tiny                  : H@1=0/38  H@5=0/38
house                 : H@1=2/2  H@5=2/2
illustration          : H@1=1/2  H@5=2/2
phylogenetic_tree     : H@1=1/1  H@5=1/1
doc_fullpage          : H@1=2/12  H@5=12/12

Micro H@1 = 79/203 = 0.389
Micro H@5 = 138/203 = 0.680


In [5]:
# 1) Extract and cache embeddings
import torch, pandas as pd, json
from PIL import Image
from tqdm import tqdm
import numpy as np
from transformers import AutoProcessor, AutoModel

MODEL = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()

DATA_CSV = "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated.csv"
SCHEMA_JSON = "kaggle/working/rag_knowledge_base/invalid_filter/labels_schema_filtered.json"
labels = json.loads(open(SCHEMA_JSON).read())
df = pd.read_csv(DATA_CSV)
rows = df[df[[f"label_{l}" for l in labels]].sum(axis=1)>=1].reset_index(drop=True)

def embed_one(path):
    try:
        im = Image.open(path).convert("RGB")
    except Exception:
        return None
    w,h = im.size
    if min(w,h) < 24:  # skip ultra tiny
        return None
    with torch.no_grad():
        inputs = processor(images=im, return_tensors="pt")
        f = model.get_image_features(**inputs)
        f = torch.nn.functional.normalize(f, dim=-1)[0].cpu().numpy()
    return f

X, Y, P = [], [], []
for _, r in tqdm(rows.iterrows(), total=len(rows)):
    f = embed_one(r["path"])
    if f is None: 
        continue
    X.append(f)
    Y.append([int(r[f"label_{l}"]) for l in labels])
    P.append(r["path"])
X = np.stack(X); Y = np.array(Y, dtype=int)

np.save("siglip_img_embeddings.npy", X)
np.save("siglip_labels.npy", Y)
json.dump(labels, open("siglip_labels_schema.json","w"))
json.dump(P, open("siglip_paths.json","w"))
print(X.shape, Y.shape)


100%|██████████| 207/207 [03:35<00:00,  1.04s/it]

(177, 1152) (177, 16)


In [6]:
# 2) Train OvR logistic regression + metrics
import numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, average_precision_score

X = np.load("siglip_img_embeddings.npy")
Y = np.load("siglip_labels.npy")
labels = json.load(open("siglip_labels_schema.json"))

# simple random split (small data -> keep seed fixed)
rng = np.random.RandomState(42)
idx = np.arange(len(X))
rng.shuffle(idx)
n_train = int(0.8*len(idx))
tr, va = idx[:n_train], idx[n_train:]

clf = OneVsRestClassifier(
    LogisticRegression(
        solver="saga", penalty="l2", C=1.0, max_iter=5000,
        class_weight="balanced", n_jobs=1
    )
)
clf.fit(X[tr], Y[tr])

probs = clf.predict_proba(X[va])        # [N, C]
preds = (probs >= 0.5).astype(int)      # naive global threshold

# Metrics
micro_f1 = f1_score(Y[va], preds, average="micro", zero_division=0)
macro_f1 = f1_score(Y[va], preds, average="macro", zero_division=0)
per_class_f1 = f1_score(Y[va], preds, average=None, zero_division=0)

aps = []
for c in range(len(labels)):
    if Y[va][:,c].sum() == 0:
        aps.append(np.nan)
    else:
        aps.append(average_precision_score(Y[va][:,c], probs[:,c]))
mAP = np.nanmean(aps)

print(f"Micro F1: {micro_f1:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"mAP:      {mAP:.3f}")
for lab, f1c, ap in zip(labels, per_class_f1, aps):
    print(f"{lab:22s}  F1={f1c:.3f}  AP={ap if ap==ap else float('nan'):.3f}")


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 7 is present in all training examples.
  warnings.warn(
/medi

Micro F1: 0.225
Macro F1: 0.352
mAP:      0.773
qr                      F1=0.364  AP=1.000
logo                    F1=0.667  AP=0.750
chart                   F1=0.000  AP=nan
journal_banner          F1=0.842  AP=0.948
pure_text               F1=0.054  AP=1.000
book_cover              F1=0.000  AP=nan
empty                   F1=0.769  AP=0.948
ecg                     F1=0.000  AP=0.028
profile                 F1=0.000  AP=nan
gel_electrophoresis     F1=0.000  AP=0.806
map                     F1=1.000  AP=1.000
tiny                    F1=0.889  AP=1.000
house                   F1=0.000  AP=nan
illustration            F1=0.054  AP=0.028
phylogenetic_tree       F1=0.000  AP=nan
doc_fullpage            F1=1.000  AP=1.000


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:
!uv  pip install iterative-stratification

Resolved 6 packages in 785ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)------------------     0 B/8.32 KiB        
Prepared 1 package in 46ms                                                   
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 7msation==0.1.9                      
 + iterative-stratification==0.1.9


In [11]:
import numpy as np, json
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, average_precision_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
import joblib

# Load cached features/labels
X = np.load("siglip_img_embeddings.npy")   # [N, D]
Y = np.load("siglip_labels.npy")           # [N, C]
labels = json.load(open("siglip_labels_schema.json"))

# Per-class threshold search to maximize F1 on val
def best_threshold(y_true, p):
    best_t, best_f1 = 0.5, 0.0
    prec, rec, thr = precision_recall_curve(y_true, p)
    grid = np.unique(np.concatenate([thr, np.linspace(0.05, 0.95, 19)]))
    for t in grid:
        yhat = (p >= t).astype(int)
        tp = (yhat & (y_true==1)).sum()
        fp = (yhat & (y_true==0)).sum()
        fn = ((1-yhat) & (y_true==1)).sum()
        if tp+fp == 0 or tp+fn == 0: 
            continue
        f1 = 2*tp / (2*tp + fp + fn)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

# Proper 5-fold CV
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
all_thresholds = []

for fold, (tr_idx, va_idx) in enumerate(mskf.split(X, Y)):
    print(f"\n--- Fold {fold+1}/5 ---")
    Xtr, Xva = X[tr_idx], X[va_idx]
    Ytr, Yva = Y[tr_idx], Y[va_idx]
    
    # Drop constant labels in TRAIN (no variation) - fixes scikit-learn warning
    var_mask = (Ytr.sum(axis=0) > 0) & (Ytr.sum(axis=0) < len(Ytr))
    keep_idx = np.where(var_mask)[0]
    Xtr_filtered = Xtr
    Ytr_filtered = Ytr[:, keep_idx]
    Xva_filtered = Xva
    Yva_filtered = Yva[:, keep_idx]
    labels_filtered = [labels[i] for i in keep_idx]
    print(f"Kept {len(keep_idx)}/{len(var_mask)} labels after dropping constants")
    
    if len(keep_idx) == 0:
        print("No valid labels in this fold, skipping...")
        continue
    
    # Optional scaling (doesn't densify)
    scaler = StandardScaler(with_mean=False)
    Xtr_scaled = scaler.fit_transform(Xtr_filtered)
    Xva_scaled = scaler.transform(Xva_filtered)
    
    # Train OvR logistic regression (stable on small data)
    clf = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",      # better than saga for tiny, imbalanced sets
            penalty="l2",
            C=2.0,
            max_iter=10000,
            class_weight="balanced",
        )
    )
    clf.fit(Xtr_scaled, Ytr_filtered)
    
    # Predict probs on val
    probs = clf.predict_proba(Xva_scaled)  # [Nva, C_filtered]
    
    # Per-class threshold search
    thresh = np.full(len(labels), 0.5)  # Default threshold for all original labels
    val_preds = np.zeros_like(Yva)
    per_class_ap = np.full(len(labels), np.nan)
    skipped = []
    
    for i, c in enumerate(keep_idx):
        lab = labels[c]
        y_true = Yva_filtered[:, i]
        if y_true.sum() == 0 or (len(y_true) - y_true.sum()) == 0:
            skipped.append(lab)
            continue
        p = probs[:, i]
        t, _ = best_threshold(y_true, p)
        thresh[c] = t
        val_preds[:, c] = (p >= t).astype(int)
        per_class_ap[c] = average_precision_score(y_true, p)
    
    # Metrics (only on labels that were trained)
    micro_f1 = f1_score(Yva_filtered, val_preds[:, keep_idx], average="micro", zero_division=0)
    macro_f1 = f1_score(Yva_filtered, val_preds[:, keep_idx], average="macro", zero_division=0)
    per_class_f1 = f1_score(Yva, val_preds, average=None, zero_division=0)
    mAP = np.nanmean(per_class_ap)
    
    print(f"Micro F1: {micro_f1:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"mAP:      {mAP:.3f}")
    
    for lab, f1c, ap, t in zip(labels, per_class_f1, per_class_ap, thresh):
        if not np.isnan(ap):
            ap_str = f"{ap:.3f}"
        else:
            ap_str = "nan"
        print(f"{lab:22s}  F1={f1c:.3f}  AP={ap_str:>6}  thr={t:.2f}")
    
    if skipped:
        print("⚠️ Skipped (no pos/neg in val):", skipped)
    
    # Store fold results
    fold_results.append({
        'micro_f1': micro_f1,
        'macro_f1': macro_f1, 
        'mAP': mAP,
        'per_class_f1': per_class_f1,
        'per_class_ap': per_class_ap,
        'thresholds': thresh.copy(),
        'keep_idx': keep_idx.copy()
    })
    all_thresholds.append(thresh.copy())

# Average across folds
if fold_results:
    avg_micro = np.mean([r['micro_f1'] for r in fold_results])
    avg_macro = np.mean([r['macro_f1'] for r in fold_results])
    avg_mAP = np.mean([r['mAP'] for r in fold_results])
    
    print(f"\n=== 5-Fold CV Results ===")
    print(f"Avg Micro F1: {avg_micro:.3f} ± {np.std([r['micro_f1'] for r in fold_results]):.3f}")
    print(f"Avg Macro F1: {avg_macro:.3f} ± {np.std([r['macro_f1'] for r in fold_results]):.3f}")
    print(f"Avg mAP:      {avg_mAP:.3f} ± {np.std([r['mAP'] for r in fold_results]):.3f}")
    
    # Build robust per-label threshold averages (only where label was trained + had pos/neg)
    L = len(labels)
    thr_sums = np.zeros(L, dtype=float)
    thr_counts = np.zeros(L, dtype=int)
    
    for r in fold_results:
        thr = r["thresholds"]      # length = L, but only meaningful on r['keep_idx']
        keep = set(r["keep_idx"].tolist())
        valid_mask = ~np.isnan(r["per_class_ap"])  # True => had pos/neg
        for i in range(L):
            if (i in keep) and valid_mask[i]:
                thr_sums[i] += thr[i]
                thr_counts[i] += 1
    
    avg_thresholds = np.full(L, 0.5, dtype=float)
    mask = thr_counts > 0
    avg_thresholds[mask] = thr_sums[mask] / thr_counts[mask]
    
    # Average per-class metrics
    print(f"\nPer-class averages:")
    avg_per_class_f1 = np.nanmean([r['per_class_f1'] for r in fold_results], axis=0)
    avg_per_class_ap = np.nanmean([r['per_class_ap'] for r in fold_results], axis=0)
    
    for i, lab in enumerate(labels):
        f1_val = avg_per_class_f1[i] if not np.isnan(avg_per_class_f1[i]) else 0.0
        ap_val = avg_per_class_ap[i]
        ap_str = f"{ap_val:.3f}" if not np.isnan(ap_val) else "nan"
        thr_info = f"({thr_counts[i]} folds)" if thr_counts[i] > 0 else "(no valid folds)"
        print(f"{lab:22s}  F1={f1_val:.3f}  AP={ap_str:>6}  thr={avg_thresholds[i]:.2f} {thr_info}")

# Train final model on all data for deployment
print(f"\n=== Training Final Model on All Data ===")
# Drop constant labels from full dataset
var_mask_full = (Y.sum(axis=0) > 0) & (Y.sum(axis=0) < len(Y))
keep_idx_full = np.where(var_mask_full)[0]
X_final = X
Y_final = Y[:, keep_idx_full]
labels_final = [labels[i] for i in keep_idx_full]

# Scale features
scaler_final = StandardScaler(with_mean=False)
X_scaled_final = scaler_final.fit_transform(X_final)

# Train final classifier
clf_final = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=2.0,
        max_iter=10000,
        class_weight="balanced",
    )
)
clf_final.fit(X_scaled_final, Y_final)

# Use average thresholds from CV - IMPORTANT: only for kept labels, in same order
final_thresholds = avg_thresholds[keep_idx_full] if fold_results else np.full(len(keep_idx_full), 0.5)

# Persist model + scaler + thresholds for reuse
joblib.dump(clf_final, "siglip_linear_probe.joblib")
joblib.dump(scaler_final, "siglip_scaler.joblib")
np.save("siglip_thresholds.npy", final_thresholds)
np.save("siglip_keep_idx.npy", keep_idx_full)  # Save which labels were kept
json.dump(labels_final, open("siglip_labels_final.json", "w"))

print("Saved: siglip_linear_probe.joblib, siglip_scaler.joblib, siglip_thresholds.npy")
print(f"Final model trained on {len(keep_idx_full)}/{len(labels)} labels")


--- Fold 1/5 ---
Kept 15/16 labels after dropping constants
Micro F1: 0.829
Macro F1: 0.658
mAP:      0.897
qr                      F1=1.000  AP= 1.000  thr=0.95
logo                    F1=0.933  AP= 0.962  thr=0.05
chart                   F1=0.000  AP=   nan  thr=0.50
journal_banner          F1=1.000  AP= 1.000  thr=0.99
pure_text               F1=1.000  AP= 1.000  thr=0.55
book_cover              F1=0.000  AP=   nan  thr=0.50
empty                   F1=1.000  AP= 1.000  thr=0.95
ecg                     F1=0.000  AP=   nan  thr=0.50
profile                 F1=0.143  AP= 0.077  thr=0.00
gel_electrophoresis     F1=0.800  AP= 0.833  thr=0.10
map                     F1=1.000  AP= 1.000  thr=0.01
tiny                    F1=1.000  AP= 1.000  thr=0.04
house                   F1=1.000  AP= 1.000  thr=0.05
illustration            F1=0.000  AP=   nan  thr=0.50
phylogenetic_tree       F1=0.000  AP=   nan  thr=0.50
doc_fullpage            F1=1.000  AP= 1.000  thr=0.05
⚠️ Skipped (no pos/neg in v

/tmp/ipykernel_3174186/3960214466.py:157: RuntimeWarning: Mean of empty slice
  avg_per_class_ap = np.nanmean([r['per_class_ap'] for r in fold_results], axis=0)


Saved: siglip_linear_probe.joblib, siglip_scaler.joblib, siglip_thresholds.npy
Final model trained on 16/16 labels


In [12]:
import os, json, math, csv
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
from PIL import Image, ImageStat, UnidentifiedImageError
from tqdm import tqdm

import joblib
import torch
from transformers import AutoProcessor, AutoModel

# ===================== CONFIG =====================
# Đường dẫn thư mục gốc ảnh cần inference (duyệt đệ quy)
ROOT_DIR = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images"

# Đầu ra CSV
OUT_CSV  = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"

# Model SigLIP: dùng hub id hoặc local snapshot directory
# - Nếu bạn muốn bắt buộc dùng local snapshot:
#   MODEL = "/media/pc1/Ubuntu/Extend_Data/ngoc/models--google--siglip-so400m-patch14-384/snapshots/<commit_hash>"
MODEL = "google/siglip-so400m-patch14-384"
LOCAL_ONLY = False  # đặt True nếu môi trường offline, cache đã có sẵn

# Artifacts đã train
PROBE_PATH     = "siglip_linear_probe.joblib"
SCALER_PATH    = "siglip_scaler.joblib"
THRESH_PATH    = "siglip_thresholds.npy"
LABELS_FINAL   = "siglip_labels_final.json"

# Ảnh hợp lệ
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# Rule prefilters
MIN_TINY_SIDE = 24          # min(width,height) < 24 -> 'tiny'
EMPTY_WHITE_RATIO = 0.985   # nếu tỷ lệ điểm ảnh rất trắng vượt ngưỡng -> 'empty'
EMPTY_STD_LUMA   = 4.0      # hoặc độ lệch chuẩn độ sáng quá thấp -> 'empty'

# Hành động gợi ý theo nhãn
QUARANTINE_LABELS = {"tiny", "empty"}   # gặp các nhãn này -> quarantine
INDEX_LABELS      = None                # hoặc để None -> mặc định index nếu có >=1 nhãn
# ==================================================


# ---------- Utilities ----------
def iter_images(root: str):
    root = Path(root)
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield str(p)

def is_tiny(im: Image.Image) -> bool:
    w, h = im.size
    return min(w, h) < MIN_TINY_SIDE

def is_empty(im: Image.Image) -> bool:
    """
    Heuristic: ảnh gần như trắng hoặc độ biến thiên độ sáng rất thấp.
    """
    # Luma
    g = im.convert("L")
    arr = np.asarray(g, dtype=np.uint8)
    # tỉ lệ pixel rất trắng
    white_ratio = float((arr >= 250).mean())
    if white_ratio >= EMPTY_WHITE_RATIO:
        return True
    # độ lệch chuẩn của độ sáng
    std = float(arr.std())
    return std <= EMPTY_STD_LUMA

# ---------- Load artifacts ----------
labels: List[str] = json.load(open(LABELS_FINAL))
clf = joblib.load(PROBE_PATH)
scaler = joblib.load(SCALER_PATH)
thr = np.load(THRESH_PATH)                         # thứ tự khớp với labels_final
assert len(thr) == len(labels), "Thresholds and labels_final length mismatch!"

# Load SigLIP
processor = AutoProcessor.from_pretrained(MODEL, local_files_only=LOCAL_ONLY)
siglip = AutoModel.from_pretrained(MODEL, local_files_only=LOCAL_ONLY)
siglip.eval()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
siglip.to(DEVICE)

@torch.no_grad()
def embed_image(path: str) -> Optional[np.ndarray]:
    try:
        im = Image.open(path).convert("RGB")
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

    # Rule: if tiny -> có thể bỏ qua embedding để tiết kiệm compute
    if is_tiny(im):
        return np.zeros((siglip.config.vision_config.hidden_size,), dtype=np.float32)

    inputs = processor(images=im, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    feats = siglip.get_image_features(**inputs)                 # [1, D]
    feats = torch.nn.functional.normalize(feats, dim=-1)[0]     # [D]
    return feats.detach().cpu().numpy()

def predict_from_feature(fvec: np.ndarray) -> Tuple[Dict[str, float], List[str]]:
    # Chuẩn hoá theo scaler đã fit
    f_scaled = scaler.transform(fvec[None, :])       # (1, D)
    probs = clf.predict_proba(f_scaled)[0]           # (C,)
    picks = [lab for lab, p in zip(labels, probs) if p >= thr[labels.index(lab)]]
    return {lab: float(p) for lab, p in zip(labels, probs)}, picks

def decide_action(picks: List[str], rules: Dict[str, bool]) -> str:
    # rules: {"tiny": bool, "empty": bool}
    if rules.get("tiny") or rules.get("empty"):
        return "quarantine"
    if INDEX_LABELS is None:
        return "index" if len(picks) > 0 else "review"
    return "index" if any(lbl in picks for lbl in INDEX_LABELS) else "review"

# ---------- Inference loop ----------
rows = []
for img_path in tqdm(list(iter_images(ROOT_DIR)), desc="Inferencing"):
    try:
        im = Image.open(img_path).convert("RGB")
    except Exception:
        rows.append({
            "path": img_path, "status": "unreadable"
        })
        continue

    # Rule prefilters
    tiny_flag  = is_tiny(im)
    empty_flag = (not tiny_flag) and is_empty(im)   # nếu tiny rồi thì khỏi check empty
    rule_picks = []
    rule_probs = {}

    if "tiny" in labels and tiny_flag:
        rule_picks.append("tiny")
        rule_probs["tiny"] = 1.0
    if "empty" in labels and empty_flag:
        rule_picks.append("empty")
        rule_probs["empty"] = 1.0

    # Nếu chỉ muốn rule quyết định và bỏ qua model cho tiny/empty:
    run_model = not (tiny_flag or empty_flag)

    if run_model:
        f = embed_image(img_path)
        if f is None:
            rows.append({"path": img_path, "status": "embed_failed"})
            continue
        probs, picks = predict_from_feature(f)
    else:
        # Không chạy model: xác suất các label khác đặt 0
        probs = {lab: 0.0 for lab in labels}
        for k, v in rule_probs.items():
            probs[k] = v
        picks = list(set(rule_picks))  # từ rules

    action = decide_action(picks, {"tiny": tiny_flag, "empty": empty_flag})

    # Ghi dòng kết quả (flatten probs)
    row = {
        "path": img_path,
        "status": "ok",
        "action": action,
        "picked_labels": ",".join(picks)
    }
    for lab in labels:
        row[f"prob_{lab}"] = probs.get(lab, 0.0)
    rows.append(row)

# ---------- Save CSV ----------
fieldnames = ["path", "status", "action", "picked_labels"] + [f"prob_{lab}" for lab in labels]
with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in rows:
        w.writerow(r)

print(f"Saved inference CSV to: {OUT_CSV}")
print(f"Total images processed: {len(rows)}")


Inferencing: 100%|██████████| 2302/2302 [02:48<00:00, 13.66it/s] 

Saved inference CSV to: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv
Total images processed: 2302


filter

In [14]:
import os
import csv
from pathlib import Path
from tqdm import tqdm
import shutil

# ==== CONFIG ====
CSV_PATH = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"

# Thư mục gốc hiện có chứa toàn bộ images (để tính đường dẫn tương đối)
IMAGES_ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")

# Thư mục đích (mới) để chứa ảnh bị lọc
DEST_ROOT   = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered")

# Chọn action nào sẽ “lọc” (ví dụ: chỉ chuyển quarantine)
ACTIONS_TO_FILTER = {"quarantine", "review"}       # có thể thêm {"review"} nếu muốn

# Move hay Copy (True = move, False = copy)
MOVE_FILES = True

# Optional: giữ đúng tree thư mục con
PRESERVE_TREE = True

# Optional: thử hardlink để nhanh & tiết kiệm dung lượng (cùng filesystem).
# Nếu hardlink fail thì fallback sang copy.
USE_HARDLINK_IF_POSSIBLE = False
# ==============


DEST_ROOT.mkdir(parents=True, exist_ok=True)

moved, skipped, missing = 0, 0, 0

with open(CSV_PATH, "r", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for r in tqdm(rows, desc="Filtering images"):
    path = r.get("path", "")
    action = r.get("action", "")
    status = r.get("status", "")

    if action not in ACTIONS_TO_FILTER:
        skipped += 1
        continue
    if status != "ok":
        missing += 1
        continue

    src = Path(path)
    if not src.exists():
        missing += 1
        continue

    # Xác định đích:
    # - Nếu PRESERVE_TREE: giữ cấu trúc tương đối so với IMAGES_ROOT
    # - Ngược lại: dồn hết vào một folder phẳng
    if PRESERVE_TREE:
        try:
            rel = src.relative_to(IMAGES_ROOT)
        except ValueError:
            # Nếu ảnh không nằm dưới IMAGES_ROOT, chỉ lấy tên file
            rel = Path(src.name)
        dst = DEST_ROOT / action / rel
    else:
        dst = DEST_ROOT / action / src.name

    dst.parent.mkdir(parents=True, exist_ok=True)

    # Di chuyển hoặc copy
    try:
        if MOVE_FILES:
            # move sẽ tự tạo overwrite? -> mặc định không. Xử lý nếu file đã tồn tại
            if dst.exists():
                # Nếu file đã tồn tại đích, đổi tên tránh ghi đè
                dst = dst.with_name(dst.stem + "__dup" + dst.suffix)
            shutil.move(str(src), str(dst))
        else:
            if USE_HARDLINK_IF_POSSIBLE:
                try:
                    os.link(src, dst)   # hardlink
                except Exception:
                    shutil.copy2(src, dst)  # fallback copy
            else:
                shutil.copy2(src, dst)
        moved += 1
    except Exception as e:
        print(f"⚠️ Lỗi khi chuyển: {src} -> {dst}: {e}")

print(f"\nDone. moved/copied: {moved}, skipped (action not in {ACTIONS_TO_FILTER}): {skipped}, missing/bad: {missing}")
print(f"Output root: {DEST_ROOT}")


Filtering images: 100%|██████████| 2302/2302 [00:00<00:00, 10371.26it/s]


Done. moved/copied: 473, skipped (action not in {'review', 'quarantine'}): 1028, missing/bad: 801
Output root: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered


In [17]:
import csv
from pathlib import Path
import shutil

# ==== CONFIG ====
CSV_PATH     = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"
IMAGES_ROOT  = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")
FILTERED_ROOT= Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered")
ACTION       = "review"     # khôi phục ảnh bị gán action này
DRY_RUN      = False         # True: chỉ in; False: move thật
# ===============

def rel_from_images_root(p: Path) -> Path:
    try:
        return p.relative_to(IMAGES_ROOT)
    except ValueError:
        return Path(p.name)  # fallback khi không cùng gốc

rows = []
with open(CSV_PATH, "r", newline="") as f:
    rows = list(csv.DictReader(f))

moved = skipped = missing = 0
for r in rows:
    if r.get("action") != ACTION or r.get("status") != "ok":
        skipped += 1
        continue

    orig_path = Path(r["path"])
    rel = rel_from_images_root(orig_path)

    # đường dẫn hiện tại của file trong filtered/
    src = FILTERED_ROOT / ACTION / rel
    # đích khôi phục là đúng đường dẫn gốc
    dst = orig_path

    if not src.exists():
        # fallback: có thể lúc move trước đó không preserve tree, thử theo tên file
        alt = FILTERED_ROOT / ACTION / src.name
        if alt.exists():
            src = alt
        else:
            print(f"⚠️ Không tìm thấy file trong filtered: {src}")
            missing += 1
            continue

    if not DRY_RUN:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            # tránh ghi đè (giữ bản khôi phục)
            dst = dst.with_name(dst.stem + "__restored" + dst.suffix)
        shutil.move(str(src), str(dst))
    else:
        print(f"[DRY] move {src} -> {dst}")

    moved += 1

print(f"\nDone. moved={moved}, skipped={skipped}, missing={missing}")
print(f"DRY_RUN={DRY_RUN} | FILTERED_ROOT={FILTERED_ROOT} | IMAGES_ROOT={IMAGES_ROOT}")



Done. moved=473, skipped=1829, missing=0
DRY_RUN=False | FILTERED_ROOT=/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered | IMAGES_ROOT=/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images


In [18]:
from pathlib import Path

root = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")
count = sum(1 for _ in root.rglob("*") if _.is_file())
print(f"Tổng số file: {count}")


Tổng số file: 1501
